## 10.3 LSTM - 前向传播案例（完整流程 + 维度计算）

#### 1、案例目标

##### 1.1 我们这一节要做什么
把「单时间步 + 多时间步 + batch + 维度」全部串起来，走一遍完整流程。

要达到的理解是：

- 每一步在算什么 ✅
- 每个张量的维度是多少 ✅
- 为什么维度是这样的 ✅
- 整个序列是怎么“滚动”起来的 ✅

##### 1.2 这一节的学习重点

- 数据是怎么从输入一路流到输出的
- 每一步的 shape 是如何变化的
- 为什么 PyTorch 需要三维输入

#### 2、案例设定

##### 2.1 我们先设定一个具体场景，所有推导都基于这个：

##### 2.2 基本参数

- batch size = 2
- 序列长度 $T = 3$
- 输入维度 $d_x = 4$
- 隐藏维度 $d_h = 5$

##### 2.3 输入数据结构

使用 PyTorch 常见格式（`batch_first=True`）：

$X \in \mathbb{R}^{2 \times 3 \times 4}$

解释：

- `2` → 两条样本
- `3` → 每条序列有 3 个时间步
- `4` → 每个时间步有 4 个特征

#### 3、整体流程

##### 3.1 时间展开结构

整个计算流程是：

$X \rightarrow LSTM \rightarrow Output$

变成：

$(x_1, x_2, x_3)$

按时间步展开依次计算：

$(x_t,\ h_{t-1},\ C_{t-1}) \rightarrow (h_t,\ C_t)$

##### 3.2 关键理解一句话

整个序列 = 单时间步计算 × 时间维展开 × batch 并行

#### 4、逐时间步详细推导（核心部分）⭐

##### 4.1 Step 1：时间步 $t = 1$

**（1）当前输入**

$x_1 \in \mathbb{R}^{2 \times 4}$

**（2）初始状态**

通常初始化为 0：

$h_0 \in \mathbb{R}^{2 \times 5}$

$C_0 \in \mathbb{R}^{2 \times 5}$

（3）四个门计算

所有门输出维度都是：

$(2 \times 5)$

分别是：

$ f_1 \in \mathbb{R}^{2 \times 5}$

$ i_1 \in \mathbb{R}^{2 \times 5}$

$ \tilde{C}_1 \in \mathbb{R}^{2 \times 5}$

$ o_1 \in \mathbb{R}^{2 \times 5}$

**（4）更新细胞状态**

$C_1 = f_1 \odot C_0 + i_1 \odot \tilde{C}_1$

维度：

$C_1 \in \mathbb{R}^{2 \times 5}$

同时：

$h_1 = o_1 \odot \tanh(C_1)$

$h_1 \in \mathbb{R}^{2 \times 5}$


##### 4.2 Step 2：时间步 $t = 2$

**（1）输入**

$x_2 \in \mathbb{R}^{2 \times 4}$

**（2）上一时刻状态**

$h_1 \in \mathbb{R}^{2 \times 5}$

$C_1 \in \mathbb{R}^{2 \times 5}$

**（3）门计算**

得到：

$ f_2,\ i_2,\ \tilde{C}_2,\ o_2 \in \mathbb{R}^{2 \times 5}$

**（4）更新状态**

$C_2 = f_2 \odot C_1 + i_2 \odot \tilde{C}_2$

$h_2 = o_2 \odot \tanh(C_2)$

其中：

$C_2,\ h_2 \in \mathbb{R}^{2 \times 5}$


##### 4.3 Step 3：时间步 $t = 3$

完全一样流程：

输入：

$x_3 \in \mathbb{R}^{2 \times 4}$

输出：

$ f_3,\ i_3,\ \tilde{C}_3,\ o_3 \in \mathbb{R}^{2 \times 5}$

$C_3,\ h_3 \in \mathbb{R}^{2 \times 5}$

#### 5、把所有时间步结果拼起来

##### 5.1 所有隐藏状态

$H = [h_1,\ h_2,\ h_3]$

维度：

$H \in \mathbb{R}^{2 \times 3 \times 5}$

解释：

- `2` → batch size
- `3` → 时间步数量
- `5` → hidden size

##### 5.2 最终状态

最终隐藏状态：

$h_3 \in \mathbb{R}^{2 \times 5}$

最终细胞状态：

$C_3 \in \mathbb{R}^{2 \times 5}$

#### 六、用表格总结整个维度变化（非常重要）📊

| 阶段 | 张量 | Shape |
|---|---|---|
| 输入整体 | $X$ | $(2,\ 3,\ 4)$ |
| 单步输入 | $x_t$ | $(2,\ 4)$ |
| 初始隐藏状态 | $h_0$ | $(2,\ 5)$ |
| 初始细胞状态 | $C_0$ | $(2,\ 5)$ |
| 四个门输出 | $f_t,\ i_t,\ \tilde{C}_t,\ o_t$ | $(2,\ 5)$ |
| 当前细胞状态 | $C_t$ | $(2,\ 5)$ |
| 当前隐藏状态 | $h_t$ | $(2,\ 5)$ |
| 所有时间步输出 | $H$ | $(2,\ 3,\ 5)$ |
| 最终隐藏状态 | $h_3$ | $(2,\ 5)$ |
| 最终细胞状态 | $C_3$ | $(2,\ 5)$ |